## Download PJM RT-HRL LMP Historical Data
                                                                                                                                                                                            
Downloads real-time hourly LMP data for **all PJM nodes** for periods older than the 731-day standard data window.
Historical data cannot be filtered by `pnode_id` at the API level — all nodes are downloaded and filtered to Virginia pnode IDs before saving.
                                                                                                                                                                                            
- Date range: January 2015 – April 2024                                                                                                                                                      
- Test month: April 2024 (run first to verify before full download)                                                                                                                          
- Stores results in DuckDB: `data/raw/pjm_lmp_data/pjm_lmp_historical.duckdb`                                                                                                                
                                                                            

In [ ]:
import requests
import pandas as pd
import duckdb
import time
from pathlib import Path
from datetime import datetime
from dateutil.relativedelta import relativedelta

DATA_DIR = next(
    p / "data" for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data").exists()
)

API_KEY = "9db34bbf6e804cd181d272bbb8329607"
headers = {"Ocp-Apim-Subscription-Key": API_KEY}
BASE_URL = "https://api.pjm.com/api/v1/rt_hrl_lmps"

In [70]:
# Test API connection                                                                                                                                                                          
r = requests.get(BASE_URL, params={                                                                                                                                                            
    "rowCount": 1,
    "startRow": 1,                                                                                                                                                                             
    "datetime_beginning_ept": "2024-01-01 00:00"
}, headers=headers)                                                                                                                                                                            
print(f"Status code: {r.status_code}")
if r.status_code == 200:                                                                                                                                                                       
    print("Connection successful")                                                                                                                                                             
    print(r.json()["items"][0])
else:                                                                                                                                                                                          
    print(f"Connection failed: {r.text[:200]}")

Status code: 200
Connection successful
{'datetime_beginning_utc': '2024-01-01T05:00:00', 'datetime_beginning_ept': '2024-01-01T00:00:00', 'pnode_id': 1, 'pnode_name': 'PJM-RTO', 'voltage': None, 'equipment': None, 'type': 'ZONE', 'zone': None, 'system_energy_price_rt': 31.14, 'total_lmp_rt': 31.197257, 'congestion_price_rt': 0.044997, 'marginal_loss_price_rt': 0.01726, 'row_is_current': True, 'version_nbr': 1}


In [ ]:
# Load Virginia pnode IDs for post-download filtering
va_pnode_ids = pd.read_csv(DATA_DIR / "processed" / "preprocessing" / "pnode_data" / "va_pnode_ids_final.csv")
va_pnode_id_set = set(va_pnode_ids["pnode_id"].tolist())
print(f"Virginia pnode IDs loaded: {len(va_pnode_id_set)}")

Virginia pnode IDs loaded: 1139


In [ ]:
# Set up DuckDB
db_path = DATA_DIR / "raw" / "pjm_lmp_data" / "pjm_lmp_historical.duckdb"
con = duckdb.connect(str(db_path))

con.execute("""
    CREATE TABLE IF NOT EXISTS rt_hrl_lmps (
        datetime_beginning_utc VARCHAR,
        datetime_beginning_ept VARCHAR,
        pnode_id BIGINT,
        pnode_name VARCHAR,
        voltage VARCHAR,
        equipment VARCHAR,
        type VARCHAR,
        zone VARCHAR,
        system_energy_price_rt DOUBLE,
        total_lmp_rt DOUBLE,
        congestion_price_rt DOUBLE,
        marginal_loss_price_rt DOUBLE,
        row_is_current BOOLEAN,
        version_nbr INTEGER
    )
""")

# Track completed months for resume capability
try:
    done_months = con.execute("""
        SELECT DISTINCT substr(datetime_beginning_ept, 1, 7) AS month
        FROM rt_hrl_lmps
    """).fetchdf()["month"].tolist()
    done_months_set = set(m for m in done_months if m)
except Exception:
    done_months_set = set()

In [73]:
# Define the range of months to fetch and loop through them
def fetch_historical_month(start_date, max_retries=5):
    """
    Download all PJM nodes for a full month (no pnode_id filter).
    Filters by type=LOAD;GEN;EHV to exclude AGGREGATE and other unused types.
    Paginates through all rows. Retries on 429 with backoff.
    Returns a DataFrame filtered to Virginia pnode_ids.
    """
    end_date = start_date + relativedelta(months=1) - relativedelta(days=1)
    date_str = (
        f"{start_date.month}-{start_date.day}-{start_date.year} 00:00 to "
        f"{end_date.month}-{end_date.day}-{end_date.year} 23:00"
    )

    all_rows = []
    start_row = 1
    page = 0

    while True:
        params = {
            "rowCount": 50000,
            "startRow": start_row,
            "datetime_beginning_ept": date_str,
            "row_is_current": 1,
            "type": "LOAD;GEN;EHV",
        }

        for attempt in range(max_retries):
            r = requests.get(BASE_URL, headers=headers, params=params)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                wait = 10 * (2 ** attempt)  # 10s, 20s, 40s, 80s, 160s
                print(f"    429 rate limit — waiting {wait}s (attempt {attempt+1}/{max_retries})", flush=True)
                time.sleep(wait)
            else:
                print(f"    API error {r.status_code}: {r.text[:200]}")
                return pd.DataFrame(all_rows) if all_rows else pd.DataFrame()
        else:
            print("    Max retries exceeded, stopping pagination")
            break

        items = r.json().get("items", [])
        all_rows.extend(items)
        page += 1

        if page % 20 == 0:
            print(f"    Page {page} — {len(all_rows):,} rows so far", flush=True)

        if len(items) < 50000:
            break
        start_row += 50000
        time.sleep(10)  # 6 connections/min limit

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    # Filter to Virginia pnode_ids only
    if "pnode_id" in df.columns:
        df = df[df["pnode_id"].isin(va_pnode_id_set)]
    return df

In [8]:
# Test download: April 2024 only
test_date = datetime(2024, 4, 1)
month_key = test_date.strftime("%Y-%m")

if month_key in done_months_set:
    print(f"{month_key} already downloaded — skipping")
else:
    print(f"Downloading {month_key} (all PJM nodes, filter after)...", flush=True)
    t0 = time.time()

    df = fetch_historical_month(test_date)

    if not df.empty:
        con.execute("INSERT INTO rt_hrl_lmps SELECT * FROM df")
        print(f"  Saved {len(df):,} Virginia rows for {month_key}")
    else:
        print(f"  No data returned for {month_key}")

    elapsed = time.time() - t0
    print(f"  Completed in {elapsed:.0f}s")

con.close()

    Page 20 — 1,000,000 rows so far
    Page 40 — 2,000,000 rows so far
    Page 60 — 3,000,000 rows so far
    Page 80 — 4,000,000 rows so far
    Page 100 — 5,000,000 rows so far
    Page 120 — 6,000,000 rows so far
    Page 140 — 7,000,000 rows so far
    Page 160 — 8,000,000 rows so far
    Page 180 — 9,000,000 rows so far
  Saved 779,760 Virginia rows for 2024-04
  Completed in 3873s


In [9]:
# Inspect results
con = duckdb.connect(str(db_path))
print("Total rows:", con.execute("SELECT COUNT(*) FROM rt_hrl_lmps").fetchone()[0])
print("\nDate range:")
print(con.execute("SELECT MIN(datetime_beginning_ept), MAX(datetime_beginning_ept) FROM rt_hrl_lmps").fetchone())
print("\nUnique pnode_ids:", con.execute("SELECT COUNT(DISTINCT pnode_id) FROM rt_hrl_lmps").fetchone()[0])
print("\nType breakdown:")
print(con.execute("SELECT type, COUNT(*) as n FROM rt_hrl_lmps GROUP BY type ORDER BY n DESC").fetchdf())
con.close()

Total rows: 779760

Date range:
('2024-04-01T00:00:00', '2024-04-30T23:00:00')

Unique pnode_ids: 1083

Type breakdown:
   type       n
0  LOAD  614160
1   GEN  139680
2   EHV   25920


In [ ]:
# Inspect a few rows
con = duckdb.connect(str(db_path))                                                                                                                                                         
con.execute("SELECT * FROM rt_hrl_lmps LIMIT 5").fetchdf()  

,datetime_beginning_utc,datetime_beginning_ept,pnode_id,pnode_name,voltage,equipment,type,zone,system_energy_price_rt,total_lmp_rt,congestion_price_rt,marginal_loss_price_rt,row_is_current,version_nbr
0,2024-04-01T04:00:00,2024-04-01T00:00:00,48907,GOSHEN,35 KV,BUS1,LOAD,PECO,15.03,15.75,0.54,0.18,True,1
1,2024-04-01T04:00:00,2024-04-01T00:00:00,48908,GOSHEN,35 KV,BUS2,LOAD,PECO,15.03,15.75,0.54,0.18,True,1
2,2024-04-01T04:00:00,2024-04-01T00:00:00,49283,ROCKRIDG,13 KV,LOAD3,LOAD,BGE,15.03,15.59,0.56,0.00,True,1
3,2024-04-01T04:00:00,2024-04-01T00:00:00,49284,ROCKRIDG,34 KV,LOAD1,LOAD,BGE,15.03,15.59,0.56,0.00,True,1
4,2024-04-01T04:00:00,2024-04-01T00:00:00,49285,ROCKRIDG,34 KV,LOAD2,LOAD,BGE,15.03,15.70,0.57,0.11,True,1


In [43]:
# Download January – March 2024                                                                                                                                                            
start_date = datetime(2024, 1, 1)                                                                                                                                                          
end_date = datetime(2024, 3, 1)                                                                                                                                                            
                                                                                                                                                                                            
total_rows = 0
current = start_date                                                                                                                                                                       
                
con = duckdb.connect(str(db_path))

while current <= end_date:
    month_key = current.strftime("%Y-%m")

    if month_key in done_months_set:                                                                                                                                                       
        print(f"Skipping {month_key} (already downloaded)")
        current += relativedelta(months=1)                                                                                                                                                 
        continue

    print(f"Downloading {month_key}...", flush=True)
    t0 = time.time()
                                                                                                                                                                                            
    df = fetch_historical_month(current)
                                                                                                                                                                                            
    if not df.empty:
        con.execute("INSERT INTO rt_hrl_lmps SELECT * FROM df")
        total_rows += len(df)

    elapsed = time.time() - t0                                                                                                                                                             
    print(f"  {month_key}: {len(df):,} rows in {elapsed:.0f}s — cumulative: {total_rows:,}", flush=True)
    current += relativedelta(months=1)                                                                                                                                                     
                
con.close()
print(f"\nDone. Total rows stored this run: {total_rows:,}")

Skipping 2024-01 (already downloaded)
Skipping 2024-02 (already downloaded)
Skipping 2024-03 (already downloaded)

Done. Total rows stored this run: 0


In [74]:
# Download each year
# Manually update with the month and year to be downloaded at a time, will then add to the DB
# Approx 1 hour per month, do one year/some months at a time to check download/API connection is working                                                                                                                                                                   
start_date = datetime(2019, 11, 1)                                                                                                                                                          
end_date = datetime(2019, 11, 1)                                                                                                                                                           
                                                                                                                                                                                            
total_rows = 0                                                                                                                                                                             
current = start_date                                                                                                                                                                       
                
con = duckdb.connect(str(db_path))                                                                                                                                                         

while current <= end_date:                                                                                                                                                                 
    month_key = current.strftime("%Y-%m")
                                                                                                                                                                                            
    if month_key in done_months_set:
        print(f"Skipping {month_key} (already downloaded)")                                                                                                                                
        current += relativedelta(months=1)                                                                                                                                                 
        continue
                                                                                                                                                                                            
    print(f"Downloading {month_key}...", flush=True)                                                                                                                                       
    t0 = time.time()
                                                                                                                                                                                            
    df = fetch_historical_month(current)                                                                                                                                                   

    if not df.empty:                                                                                                                                                                       
        con.execute("INSERT INTO rt_hrl_lmps SELECT * FROM df")
        total_rows += len(df)                                                                                                                                                              

    elapsed = time.time() - t0                                                                                                                                                             
    print(f"  {month_key}: {len(df):,} rows in {elapsed:.0f}s — cumulative: {total_rows:,}", flush=True)
    current += relativedelta(months=1)                                                                                                                                                     
                                                                                                                                                                                            
con.close()                                                                                                                                                                                
print(f"\nDone. Total rows stored this run: {total_rows:,}")  

    Page 20 — 1,000,000 rows so far
    Page 40 — 2,000,000 rows so far
    Page 60 — 3,000,000 rows so far
    Page 80 — 4,000,000 rows so far
    Page 100 — 5,000,000 rows so far
    Page 120 — 6,000,000 rows so far
    Page 140 — 7,000,000 rows so far
    Page 160 — 8,000,000 rows so far
  2019-11: 723,884 rows in 3589s — cumulative: 723,884

Done. Total rows stored this run: 723,884


In [85]:
#Check downloads
pd.set_option('display.max_rows', None)  

con = duckdb.connect(str(db_path))                                                                                                                                                         
print("Total rows:", con.execute("SELECT COUNT(*) FROM rt_hrl_lmps").fetchone()[0])                                                                                                        
print("\nRows per month:")                                                                                                                                                                 
print(con.execute("""                                                                                                                                                                      
    SELECT substr(datetime_beginning_ept, 1, 7) AS month, COUNT(*) as rows                                                                                                                 
    FROM rt_hrl_lmps                                                                                                                                                                       
    GROUP BY month
    ORDER BY month                                                                                                                                                                         
""").fetchdf()) 
print("\nUnique pnode_ids per month:")                                                                                                                                                     
print(con.execute("""                                                                                                                                                                      
    SELECT substr(datetime_beginning_ept, 1, 7) AS month, COUNT(DISTINCT pnode_id) as pnode_ids
    FROM rt_hrl_lmps                                                                                                                                                                       
    GROUP BY month
    ORDER BY month                                                                                                                                                                         
""").fetchdf())
con.close()  

Total rows: 81024382

Rows per month:
       month    rows
0    2015-01  656208
1    2015-02  592704
2    2015-03  656334
3    2015-04  636480
4    2015-05  660576
5    2015-06  643680
6    2015-07  665136
7    2015-08  665136
8    2015-09  643824
9    2015-10  669600
10   2015-11  648900
11   2015-12  674016
12   2016-01  675552
13   2016-02  631968
14   2016-03  676324
15   2016-04  657360
16   2016-05  679992
17   2016-06  661680
18   2016-07  683736
19   2016-08  683736
20   2016-09  662904
21   2016-10  685968
22   2016-11  664762
23   2016-12  688560
24   2017-01  690432
25   2017-02  623616
26   2017-03  696116
27   2017-04  676800
28   2017-05  699864
29   2017-06  678960
30   2017-07  701592
31   2017-08  701592
32   2017-09  683448
33   2017-10  709776
34   2017-11  687834
35   2017-12  710568
36   2018-01  712008
37   2018-02  643104
38   2018-03  714315
39   2018-04  694800
40   2018-05  719256
41   2018-06  699120
42   2018-07  722424
43   2018-08  722424
44   2018-09  701

In [86]:
#Check for duplicates
con = duckdb.connect(str(db_path))
con.execute("""
    SELECT COUNT(*) as duplicate_pairs      
    FROM (                                  
        SELECT datetime_beginning_utc, pnode_id
        FROM rt_hrl_lmps                                                                                                                                                                   
        GROUP BY datetime_beginning_utc, pnode_id                                                                                                                                          
        HAVING COUNT(*) > 1                                                                                                                                                                
    )                                                                                                                                                                                      
""").fetchone() 

(0,)

In [ ]:
#Backup this raw data
con = duckdb.connect(str(db_path))
con.execute(f"""
    COPY rt_hrl_lmps TO '{DATA_DIR / "raw" / "pjm_lmp_data" / "pjm_lmp_historical_raw.parquet"}' (FORMAT PARQUET)
""")
print("Backup saved")
con.close()

Backup saved
